# 3. 用FastMCP创建自己的服务

上一节把一个加法函数注册到 MCP 服务，再适配成 LangChain 工具。这一节站在**工具提供方**的角度，创建一个包含加法和除法的计算服务。

服务如何知道有哪些工具？客户端如何知道参数必填、类型和默认值？参数不合法时，会不会执行函数？我们用图和调用记录逐步验证。

学习目标：

1. 创建 FastMCP 服务，用 `@mcp.tool` 注册两个业务函数。
2. 从客户端发现的 Schema 观察类型注解、默认值与范围约束。
3. 通过进程内连接调用工具，区分参数校验与业务判断。
4. 整理成独立脚本，复用同一工具定义，并验证脚本能被客户端启动和调用。

本节完全在本地运行，不需要聊天模型、API Key 或外部服务。先选择项目的 `.venv` 内核，再按顺序执行。第四节会详细讲解 stdio 与 HTTP 连接。

阅读中涉及的 Client 与 Agent 关系、关键接口和脚本执行方式，在 **3.7 答疑**中集中说明。

## 3.1 环境准备

继续使用项目已有的 `langchain[mcp]` 依赖，其中已包含 FastMCP。这里只用 FastMCP 客户端与服务端；`Annotated` 和 `Field` 用来补充参数说明及校验规则。

下面的路径定位同时支持从项目根目录和章节目录运行。Jupyter 已有事件循环，客户端调用使用顶层 `await`。

In [1]:
import json
import runpy
import sys
from importlib.metadata import version
from pathlib import Path
from typing import Annotated

from fastmcp import Client, FastMCP
from fastmcp.client.transports import PythonStdioTransport
from pydantic import Field

PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("请从 lang-chain-learn 项目或其子目录运行本笔记。")

SCRIPT_FILE = PROJECT_ROOT / "Chapter11_MCP" / "servers" / "calculator_server.py"

print("Python:", sys.executable)
for package in ["fastmcp", "mcp", "pydantic"]:
    print(f"{package}: {version(package)}")
print("服务脚本存在:", SCRIPT_FILE.is_file())

Python: /Users/skk/Developer/lang-chain-learn/.venv/bin/python
fastmcp: 4.1.0
mcp: 2.3.0
pydantic: 2.13.5
服务脚本存在: True


## 3.2 创建服务，并注册两个工具

![FastMCP 的工具注册与发现](assets/07-fastmcp-tool-registration.svg)

### 3.2.1 服务对象和加法工具

`FastMCP(name="Calculator")` 创建服务对象，`Calculator` 是服务名称。一个服务可以提供多个工具；`add`、`divide` 才是客户端调用时使用的工具名称。

`@calc_server.tool` 在定义函数时完成注册。本例默认使用函数名作为工具名、docstring 作为描述。它与第五章的 LangChain `@tool` 用途不同：这里把函数登记到 MCP 服务，后续由客户端发现和调用。

增加 `function_entries`，在函数第一行记录一次“进入函数”。它只用于观察实验，独立服务脚本会去掉这份记录。

In [2]:
calc_server = FastMCP(name="Calculator")
function_entries = []


@calc_server.tool
def add(a: int, b: int) -> int:
    """计算两个整数之和。"""
    function_entries.append({"tool": "add", "a": a, "b": b})
    return a + b

@calc_server.tool
def multiply(a: int, b: int) -> int:
    """计算两个整数之积。"""
    function_entries.append({"tool": "multiply", "a": a, "b": b})
    return a * b

print("服务名称:", calc_server.name)
print("服务的工具:",await calc_server.list_tools())
print("注册加法后，函数进入记录:", function_entries)

服务名称: Calculator
服务的工具: [FunctionTool(name='add', title=None, description='计算两个整数之和。', tags=set()), FunctionTool(name='multiply', title=None, description='计算两个整数之积。', tags=set())]
注册加法后，函数进入记录: []


### 3.2.2 除法工具：默认值、参数约束和业务判断

除法需要两个数，还允许指定保留的小数位数。`precision=2` 表示省略该参数时使用 2；`Field(ge=0, le=6)` 要求精度在 0 到 6 之间。

`Annotated[int, Field(...)]` 可以拆开看：**参数类型是 int，Field 补充描述与约束**。可回看 [第五章的参数 Schema](../Chapter05_tools/04_tools_args_schema.ipynb) 和 [第六章的 Pydantic](../Chapter06_structure_output/02-Pydantic的高级特性.ipynb)。

除数不能为 0 的判断写在函数体内。后面通过实验观察它与参数约束的区别。

In [3]:
@calc_server.tool
def divide(
    numerator: float,
    denominator: float,
    precision: Annotated[
        int, Field(ge=0, le=6, description="保留的小数位数，范围为 0 到 6。")
    ] = 2,
) -> float:
    """计算分子除以分母，并按指定精度四舍五入。"""
    function_entries.append({
        "tool": "divide",
        "numerator": numerator,
        "denominator": denominator,
        "precision": precision,
    })
    if denominator == 0:
        raise ValueError("除数不能为 0")
    return round(numerator / denominator, precision)


print("注册两个工具后，函数进入记录:", function_entries)

注册两个工具后，函数进入记录: []


这两个工具都用普通 `def` 编写。FastMCP 支持同步和异步业务函数；客户端的 `await` 用于等待 MCP 交互，并不要求加法和除法改成 `async def`。

先从**服务端**取出注册对象，确认工具已经登记。这里的 `get_tool(name)` 返回服务端工具对象；客户端的 `list_tools()` 返回什么，下一步再看。

In [4]:
registered_add = await calc_server.get_tool("add")
registered_divide = await calc_server.get_tool("divide")
# 返回值是FunctionTool
registered_multiply = await calc_server.get_tool("multiply")

for registered in [registered_add, registered_divide,registered_multiply]:
    print("服务端注册对象:", type(registered).__name__, registered.name)
print("加法工具保存原业务函数:", registered_add.fn is add)
print("当前 add 名称指向的类型:", type(add).__name__)
print("读取注册对象后，函数进入次数:", len(function_entries))

服务端注册对象: FunctionTool add
服务端注册对象: FunctionTool divide
服务端注册对象: FunctionTool multiply
加法工具保存原业务函数: True
当前 add 名称指向的类型: function
读取注册对象后，函数进入次数: 0


当前环境中，服务端注册对象是 `FunctionTool`，`add` 仍是原 Python 函数。它与 LangChain 的 `StructuredTool` 属于不同框架的对象；不要根据名字把两者混为一谈。

本节通过客户端调用来验证 MCP 的参数校验与结果包装。直接调用 `add(2, 3)` 只会走 Python 函数本身。

## 3.3 进程内发现工具，查看参数Schema

### 3.3.1 客户端得到的是描述数据

`Client(calc_server)` 使用进程内连接，不启动子进程或 HTTP 端口。进入 `async with` 后，客户端建立会话，`await client.list_tools()` 发现工具；退出上下文后关闭会话。

与第二节一样，这里返回的是 MCP 工具描述列表。

In [5]:
client = Client(calc_server)

async with client:#客户端通过协议拿到"描述"
    tool_specs = await client.list_tools()

specs_by_name = {spec.name: spec for spec in tool_specs}

print("返回容器:", type(tool_specs).__name__)
for spec in tool_specs:
    print("描述类型:", f"{type(spec).__module__}.{type(spec).__name__}")
    print("工具名称:", spec.name)
    print("工具描述:", spec.description)
print("发现后，函数进入次数:", len(function_entries))

返回容器: list
描述类型: mcp_types._types.Tool
工具名称: add
工具描述: 计算两个整数之和。
描述类型: mcp_types._types.Tool
工具名称: multiply
工具描述: 计算两个整数之积。
描述类型: mcp_types._types.Tool
工具名称: divide
工具描述: 计算分子除以分母，并按指定精度四舍五入。
发现后，函数进入次数: 0


### 3.3.2 函数签名怎样变成输入Schema

![类型注解、默认值和 Field 对应的 Schema](assets/08-fastmcp-parameter-schema.svg)

先看原始 Schema，再核对函数中的每个条件。`input_schema` 是客户端填写 `arguments` 的依据。

In [6]:
for name in ["add", "divide","multiply"]:
    print(f"{name} 的 类型是{type(specs_by_name[name])}")
    print(f"{name} 的 input_schema:")
    # 名称与描述对象的映射由前面的 specs_by_name 字典推导式创建
    print(json.dumps(specs_by_name[name].input_schema, ensure_ascii=False, indent=2))
    print()

add 的 类型是<class 'mcp_types._types.Tool'>
add 的 input_schema:
{
  "type": "object",
  "additionalProperties": false,
  "properties": {
    "a": {
      "type": "integer"
    },
    "b": {
      "type": "integer"
    }
  },
  "required": [
    "a",
    "b"
  ]
}

divide 的 类型是<class 'mcp_types._types.Tool'>
divide 的 input_schema:
{
  "type": "object",
  "additionalProperties": false,
  "properties": {
    "numerator": {
      "type": "number"
    },
    "denominator": {
      "type": "number"
    },
    "precision": {
      "default": 2,
      "description": "保留的小数位数，范围为 0 到 6。",
      "maximum": 6,
      "minimum": 0,
      "type": "integer"
    }
  },
  "required": [
    "numerator",
    "denominator"
  ]
}

multiply 的 类型是<class 'mcp_types._types.Tool'>
multiply 的 input_schema:
{
  "type": "object",
  "additionalProperties": false,
  "properties": {
    "a": {
      "type": "integer"
    },
    "b": {
      "type": "integer"
    }
  },
  "required": [
    "a",
    "b"
  ]
}



| 函数定义 | 客户端看到的 Schema | 意义 |
|---|---|---|
| `a: int`、`b: int` | `type: integer` | 加法参数是整数 |
| `numerator: float`、`denominator: float` | `type: number` | 除法接收数值，允许整数和小数 |
| 参数没有默认值 | 参数名出现在 `required` 中 | 调用时必须提供 |
| `precision = 2` | `default: 2`，不在 `required` 中 | 省略时使用 2 |
| `Field(ge=0, le=6)` | `minimum: 0`、`maximum: 6` | 限定精度范围 |
| `Field(description=...)` | 参数的 `description` | 说明这个参数的用途 |

`properties` 列出参数；本例还生成 `additionalProperties: false`，表示不接受未定义的参数名。注意：**函数体中的除数检查没有自动变成 Schema 约束**。

抽取除法 Schema 的几个字段，让对应关系更清楚。

In [7]:
divide_schema = specs_by_name["divide"].input_schema
precision_schema = divide_schema["properties"]["precision"]

print("必填参数:", divide_schema["required"])
print("precision 必填:", "precision" in divide_schema["required"])
print("precision 默认值:", precision_schema["default"])
print("precision 范围:", precision_schema["minimum"], "到", precision_schema["maximum"])
print("precision 描述:", precision_schema["description"])
print("denominator 的 Schema:", divide_schema["properties"]["denominator"])
print("查看 Schema 后，函数进入次数:", len(function_entries))

必填参数: ['numerator', 'denominator']
precision 必填: False
precision 默认值: 2
precision 范围: 0 到 6
precision 描述: 保留的小数位数，范围为 0 到 6。
denominator 的 Schema: {'type': 'number'}
查看 Schema 后，函数进入次数: 0


## 3.4 进程内调用：先加法，再除法

### 3.4.1 工具名称和业务参数分别传入

客户端调用的核心接口可以读成：

```python
await client.call_tool(name="工具名称", arguments={"业务参数": "参数值"})
```

| 参数 / 返回值 | 本节的含义 |
|---|---|
| `name` | 注册的工具名称，例如 `add` |
| `arguments` | 符合该工具输入 Schema 的参数字典 |
| 等待后的返回值 | FastMCP 客户端的 `CallToolResult`，包含结果内容与状态 |

先调用加法，观察发现与调用的区别。

In [8]:
async with client:
    add_result = await client.call_tool(name="add", arguments={"a": 2, "b": 3})

print("返回对象类型:", type(add_result).__name__)
print("is_error:", add_result.is_error)
print("计算结果 data:", add_result.data)
print("函数进入记录:", function_entries)


print("===="*10)
async with client:
    multiply_result = await client.call_tool(name="multiply", arguments={"a": 4, "b": 5})
print("返回对象类型:",type(multiply_result))
print("乘法结果:", multiply_result.data)
print("函数进入记录:", function_entries)

返回对象类型: CallToolResult
is_error: False
计算结果 data: 5
函数进入记录: [{'tool': 'add', 'a': 2, 'b': 3}]
返回对象类型: <class 'fastmcp.client.client.CallToolResult'>
乘法结果: 20
函数进入记录: [{'tool': 'add', 'a': 2, 'b': 3}, {'tool': 'multiply', 'a': 4, 'b': 5}]


### 3.4.2 修改一个条件：省略精度与指定精度

计算 `1 / 3`。第一次不传 `precision`，第二次指定 `precision=4`。先预测两个结果，再执行。

In [9]:
async with client:
    default_divide_result = await client.call_tool(
        name="divide", arguments={"numerator": 1, "denominator": 3}
    )
    precise_divide_result = await client.call_tool(
        name="divide", arguments={"numerator": 1, "denominator": 3, "precision": 4}
    )

print("省略 precision:", default_divide_result.data)
print("指定 precision=4:", precise_divide_result.data)
print("最新两次函数进入记录:")
for entry in function_entries[-2:]:
    print(entry)

省略 precision: 0.33
指定 precision=4: 0.3333
最新两次函数进入记录:
{'tool': 'divide', 'numerator': 1.0, 'denominator': 3.0, 'precision': 2}
{'tool': 'divide', 'numerator': 1.0, 'denominator': 3.0, 'precision': 4}


省略精度时，函数记录中的 `precision` 已经是 2。默认值由服务端补齐，不需要客户端提前填好。

这里使用 Python 的 `round`，返回的是数值；例如 `0.5` 不会为了精度 2 而在文本中补成 `0.50`。

### 3.4.3 返回类型与结果包装

`-> int`、`-> float` 帮助 FastMCP 生成输出 Schema。本例的基础数值结果被包装在 `result` 字段中。

`data` 是 FastMCP 客户端提供的 Python 值视图；`content` 是 MCP 内容块；`structured_content` 是结构化结果。它们表示同一次计算的不同形式。

In [10]:
print("divide 的 output_schema:")
print(json.dumps(specs_by_name["divide"].output_schema, ensure_ascii=False, indent=2))
print("data:", precise_divide_result.data)
print("content:", [block.model_dump() for block in precise_divide_result.content])
print("structured_content:", precise_divide_result.structured_content)
print("加法的结构化结果:", add_result.structured_content)

divide 的 output_schema:
{
  "properties": {
    "result": {
      "type": "number"
    }
  },
  "required": [
    "result"
  ],
  "type": "object",
  "x-fastmcp-wrap-result": true
}
data: 0.3333
content: [{'type': 'text', 'text': '0.3333', 'annotations': None, 'meta': None}]
structured_content: {'result': 0.3333}
加法的结构化结果: {'result': 5}


## 3.5 小实验：失败发生在参数校验，还是函数内部

![MCP 调用中的参数校验与业务判断](assets/09-fastmcp-call-validation.svg)

试三种输入：加法缺少 `b`、除法精度为 7、除数为 0。前两种违反参数规则，第三种满足参数类型要求，但不满足业务条件。

这里设置 `raise_on_error=False`，让预期的工具错误作为结果返回，便于连续观察。其默认值是 `True`，工具失败时客户端会抛出异常。

看两个证据：返回结果的 `is_error`，以及函数进入记录增加了几次。服务端可能同时打印错误日志，这是本实验故意触发的失败。

In [11]:
error_cases = [
    ("缺少必填参数 b", "add", {"a": 2}),
    ("精度超出范围", "divide", {"numerator": 1, "denominator": 3, "precision": 7}),
    ("除数为 0", "divide", {"numerator": 1, "denominator": 0}),
]
error_observations = []

async with client:
    for label, name, arguments in error_cases:
        count_before = len(function_entries)
        result = await client.call_tool(
            name=name, arguments=arguments, raise_on_error=False
        )
        entered_count = len(function_entries) - count_before
        error_observations.append({
            "case": label, "is_error": result.is_error, "entered_count": entered_count
        })
        print(f"{label}: is_error={result.is_error}, 函数进入次数增加={entered_count}")
        for block in result.content:
            print("错误内容:", block.text)
        print()

[10/11/26 10:08:02] WARNING  Invalid arguments for tool 'add': {'error_count': 1, 'error_types':     ]8;id=11803419;file:///Users/skk/Developer/lang-chain-learn/.venv/lib/python3.11/site-packages/fastmcp/server/server.py\server.py]8;;\:]8;id=11803420;file:///Users/skk/Developer/lang-chain-learn/.venv/lib/python3.11/site-packages/fastmcp/server/server.py#1471\1471]8;;\
                             ['missing_argument']}                                                                 

缺少必填参数 b: is_error=True, 函数进入次数增加=0
错误内容: 1 validation error for call[add]
b
  Missing required argument [type=missing_argument, input_value={'a': 2}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing_argument



                    WARNING  Invalid arguments for tool 'divide': {'error_count': 1, 'error_types':  ]8;id=11803427;file:///Users/skk/Developer/lang-chain-learn/.venv/lib/python3.11/site-packages/fastmcp/server/server.py\server.py]8;;\:]8;id=11803428;file:///Users/skk/Developer/lang-chain-learn/.venv/lib/python3.11/site-packages/fastmcp/server/server.py#1471\1471]8;;\
                             ['less_than_equal']}                                                                  

精度超出范围: is_error=True, 函数进入次数增加=0
错误内容: 1 validation error for call[divide]
precision
  Input should be less than or equal to 6 [type=less_than_equal, input_value=7, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal



[10/11/26 10:08:02] Error calling tool 'divide'                                                                    
                    ╭───────────────────────────── Traceback (most recent call last) ─────────────────────────────╮
                    │ /Users/skk/Developer/lang-chain-learn/.venv/lib/python3.11/site-packages/fastmcp/server/ser │
                    │ ver.py:1467 in call_tool                                                                    │
                    │                                                                                             │
                    │ /Users/skk/Developer/lang-chain-learn/.venv/lib/python3.11/site-packages/fastmcp/tools/base │
                    │ .py:440 in _run                                                                             │
                    │                                                                                             │
                    │                                  ... 10 frames hidden ...                                   │
                    │                                                                                             │
                    │ /Users/skk/Developer/lang-chain-learn/.venv/lib/python3.11/site-packages/fastmcp/tools/func │
                    │ tion_tool.py:96 in wrapper                                                                  │
                    │                                                                                             │
                    │ in divide:17                                                                                │
                    │                                                                                             │
                    │   14 │   │   "precision": precision,                                                        │
                    │   15 │   })                                                                                 │
                    │   16 │   if denominator == 0:                                                               │
                    │ ❱ 17 │   │   raise ValueError("除数不能为 0")                                               │
                    │   18 │   return round(numerator / denominator, precision)                                   │
                    │   19                                                                                        │
                    │   20                                                                                        │
                    ╰─────────────────────────────────────────────────────────────────────────────────────────────╯
                    ValueError: 除数不能为 0

除数为 0: is_error=True, 函数进入次数增加=1
错误内容: Error calling tool 'divide': 除数不能为 0



In [12]:
print("失败发生的位置:")
for observation in error_observations:
    print(observation)
print("最后一次函数进入记录:", function_entries[-1])

失败发生的位置:
{'case': '缺少必填参数 b', 'is_error': True, 'entered_count': 0}
{'case': '精度超出范围', 'is_error': True, 'entered_count': 0}
{'case': '除数为 0', 'is_error': True, 'entered_count': 1}
最后一次函数进入记录: {'tool': 'divide', 'numerator': 1.0, 'denominator': 0.0, 'precision': 2}


缺少参数和精度越界都没有进入函数；除数为 0 进入了函数，再由 `ValueError` 拒绝计算。Schema 描述参数格式与已声明的约束，函数体实现实际业务规则。

工具错误属于调用结果，不会移除已注册的工具。修正除数，再调用一次。

Schema只检查参数是否满足pydantic规则

In [13]:
async with client:
    recovered_result = await client.call_tool(
        name="divide", arguments={"numerator": 1, "denominator": 2}
    )

print("修正后的 is_error:", recovered_result.is_error)
print("修正后的结果:", recovered_result.data)
print("最新函数进入记录:", function_entries[-1])

修正后的 is_error: False
修正后的结果: 0.5
最新函数进入记录: {'tool': 'divide', 'numerator': 1.0, 'denominator': 2.0, 'precision': 2}


## 3.6 整理为独立服务脚本

![进程内连接与独立服务脚本](assets/10-fastmcp-inprocess-and-script.svg)

### 3.6.1 把已验证的定义移入脚本

项目中已保存 [servers/calculator_server.py](servers/calculator_server.py)。它保留同样的服务名、两个工具、Schema 约束与除数检查，去掉 notebook 的观察记录。

脚本最后增加启动入口。main() 解析启动选项，再调用 mcp.run；不提供选项时仍使用 stdio。启动选项在第四节展开：

```python
if __name__ == "__main__":
    main()
```

| 代码 | 作用 |
|---|---|
| `if __name__ == "__main__"` | 作为主程序执行时启动服务；被读取或导入时只定义和注册工具 |
| `mcp.run(...)` | 启动服务并等待客户端请求，通常持续运行 |
| `transport="stdio"` | 通过标准输入和标准输出与客户端通信 |
| `show_banner=False` | 关闭启动横幅，使实验输出更简洁 |

下面读取实际脚本，便于直接核对。运行 notebook 不会覆盖脚本文件。

In [14]:
print(SCRIPT_FILE.read_text(encoding="utf-8"))

"""第三、四节的计算 MCP 服务：支持 stdio 与 Streamable HTTP。"""

import argparse
import logging
import os
import sys
from typing import Annotated

from fastmcp import FastMCP
from pydantic import Field


mcp = FastMCP(name="Calculator")
logger = logging.getLogger("calculator")


@mcp.tool
def add(a: int, b: int) -> int:
    """计算两个整数之和。"""
    logger.info("add: a=%s b=%s", a, b)
    return a + b


@mcp.tool
def divide(
    numerator: float,
    denominator: float,
    precision: Annotated[
        int, Field(ge=0, le=6, description="保留的小数位数，范围为 0 到 6。")
    ] = 2,
) -> float:
    """计算分子除以分母，并按指定精度四舍五入。"""
    logger.info(
        "divide: numerator=%s denominator=%s precision=%s",
        numerator,
        denominator,
        precision,
    )
    if denominator == 0:
        raise ValueError("除数不能为 0")
    return round(numerator / denominator, precision)


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--transport", choices=["stdio", "http"]

### 3.6.2 先复用脚本中的对象，继续做进程内验证

`runpy.run_path` 执行脚本的顶层代码并返回对象字典，其默认的 `__name__` 是 `<run_path>`，所以不会进入启动分支。取得 `mcp` 后，仍可用 `Client(mcp)` 验证。

比较脚本中各工具的输入 Schema 与 notebook 中的对应定义。notebook 新增的乘法练习尚未移入脚本，本段核对脚本实际提供的加法与除法；runpy 的原理见 3.7.3。

In [15]:
# 运行Sever端脚本
script_objects = runpy.run_path(str(SCRIPT_FILE))
script_server = script_objects["mcp"]

async with Client(script_server) as script_client:
    script_specs = await script_client.list_tools()
    script_divide_result = await script_client.call_tool(
        name="divide", arguments={"numerator": 1, "denominator": 3, "precision": 4}
    )

# 工具名: 输入json
script_schemas = {spec.name: spec.input_schema for spec in script_specs}
# 
reference_schemas = {name: specs_by_name[name].input_schema for name in script_schemas}

print("脚本读取时的 __name__:", script_objects["__name__"])
print("脚本服务名称:", script_server.name)
print("脚本提供的工具:", sorted(script_schemas))
print("脚本工具的输入 Schema 与 notebook 对应工具一致:", script_schemas == reference_schemas)
print("脚本对象的除法结果:", script_divide_result.data)

脚本读取时的 __name__: <run_path>
脚本服务名称: Calculator
脚本提供的工具: ['add', 'divide']
脚本工具的输入 Schema 与 notebook 对应工具一致: True
脚本对象的除法结果: 0.3333


### 3.6.3 最后让客户端启动脚本，验证独立运行

此时只把连接目标从服务对象换成脚本。`PythonStdioTransport` 使用当前内核的 Python 启动子进程，保证服务与 notebook 使用同一个环境。

| 参数 | 本实验传入的内容 |
|---|---|
| `script_path` | 计算服务脚本的路径 |
| `python_cmd` | `sys.executable`，当前内核的 Python |
| `cwd` | 项目根目录，子进程的工作目录 |
| `keep_alive=False` | 客户端关闭连接时结束服务子进程 |

进入客户端上下文时自动启动服务；退出后关闭连接和子进程。**不要先在 notebook 中直接执行 `mcp.run()`**，它会持续等待请求。下面用客户端完成启动、发现、调用和清理。

In [16]:
# 创建子进程运行 server脚本，然后返回连接配置
transport = PythonStdioTransport(
    script_path=SCRIPT_FILE,
    python_cmd=sys.executable,
    cwd=str(PROJECT_ROOT),
    keep_alive=False,
)

async with Client(transport, timeout=15) as process_client:
    process_specs = await process_client.list_tools()
    process_add_result = await process_client.call_tool(
        name="add", arguments={"a": 2, "b": 3}
    )
    process_divide_result = await process_client.call_tool(
        name="divide", arguments={"numerator": 1, "denominator": 3, "precision": 4}
    )

process_schemas = {spec.name: spec.input_schema for spec in process_specs}
checks = {
    "发现两个工具": set(process_schemas) == {"add", "divide"},
    "输入 Schema 与进程内一致": process_schemas == reference_schemas,
    "加法调用成功并返回 5": not process_add_result.is_error and process_add_result.data == 5,
    "除法调用成功并返回 0.3333":
        not process_divide_result.is_error and process_divide_result.data == 0.3333,
}
for label, passed in checks.items():
    print(f"{label}: {passed}")
assert all(checks.values()), checks
print("独立服务验证完成，已退出客户端上下文。")

发现两个工具: True
输入 Schema 与进程内一致: True
加法调用成功并返回 5: True
除法调用成功并返回 0.3333: True
独立服务验证完成，已退出客户端上下文。


两种连接发现了相同的工具定义，并得到相同计算结果。独立运行时，业务函数在服务子进程执行，不会更新 notebook 中的 `function_entries`。

如需观察手动启动，在**项目根目录的终端**执行：

```bash
uv run python Chapter11_MCP/servers/calculator_server.py
```

stdio 服务会等待协议客户端请求，不会自动显示计算结果；手动启动后按 `Ctrl+C` 关闭。上面的 notebook 实验由客户端管理子进程，不需要先运行这条命令。stdio 的标准输出用于协议通信，服务端调试日志应写到标准错误；第四节继续展开连接和日志的细节。

## 3.7 答疑：Client、Agent和脚本连接怎样串起来

这一小节回答三个阅读问题：为什么先学 Client、关键接口怎样读、`runpy` 与 `PythonStdioTransport` 分别做什么。

先统一名称：本笔记使用 **`Client.call_tool`（单数）** 和 **`runpy.run_path`**。`runpy` 是 Python 标准库模块；`PythonStdioTransport` 是 FastMCP 的客户端连接类。

### 3.7.1 MCP是给Agent用的，为什么这里先调用Client

我们的学习目标确实是让 Agent 使用 MCP 工具。接入 Agent 后，**模型提出工具名和参数，Agent 的工具执行环节通过 Client 向服务发请求**。`MCPAdapter` 将这一层包装成 LangChain 工具。

![手工调用与 Agent 调用共同经过 MCP Client](assets/11-client-and-agent-tool-paths.svg)

| 使用方式 | 谁填写工具名和参数 | 谁执行业务函数 |
|---|---|---|
| 直接调用 `Client.call_tool` | 我们在代码中填写 | MCP 服务端 |
| Agent 使用适配后的工具 | 模型生成 ToolCall，Agent 按调用执行 | 同样是 MCP 服务端 |

**MCPAdapter 没有替代 Client，而是建在 Client 之上。** 当前环境（langchain 1.4.3，`langchain.mcp` 仍是 beta）的源码中，适配分三步：

1. `MCPAdapter(target)` 把目标交给 FastMCP 的 `Client(target)`，传入服务对象、Transport 或 URL 都一样；`adapter.client` 可以取回这个 Client。
2. `await adapter.list_tools()` 先调用 `client.list_tools()`，再把每个工具描述转换成 `StructuredTool`。
3. 工具被执行时，内部调用 `client.call_tool(工具名, 参数, raise_on_error=False)`，再把 `CallToolResult` 转换成 `ToolMessage`。

所以 3.3–3.5 在 Client 层观察到的内容，接入 Agent 后都还在，只是换成了 LangChain 的形式：

| Client 层（3.3–3.5 观察） | 适配后的 LangChain 工具（Agent 使用） |
|---|---|
| `client.list_tools()` 返回的工具名、描述 | `StructuredTool` 的 `name`、`description`，供模型选择工具 |
| 工具描述的 `input_schema` | `StructuredTool.args_schema`，供模型填写参数 |
| `client.call_tool(name, arguments)` | 工具执行时调用的协程 |
| `CallToolResult.content` | `ToolMessage` 的文本，也就是模型读到的结果 |
| `CallToolResult.structured_content` | `ToolMessage.artifact["structured_content"]` |
| `is_error=True`（参数校验失败或函数抛出异常） | `status="error"` 的 ToolMessage，文本是错误说明 |

`args_schema` 只是 JSON Schema 字典，LangChain 不在本地重新校验参数。例如精度为 7 的调用仍会发到服务端，由服务端拒绝，与 3.5 的结论一致。

因此，先直接调用 Client，能单独验证工具注册、Schema 和计算结果。随后接入 Agent，只新增模型选择工具与参数的环节；这样更容易判断问题发生在服务、连接，还是模型选择上。实际项目中，通常使用适配后的工具，不必为每个工具手写一次 `client.call_tool`。

**“进程内连接”说明 Client 与 Server 位于同一个 Python 进程中**，例如同一个 notebook 内核。`Client(calc_server)` 通过内存中的 MCP 连接请求服务，仍有发现、参数校验和结果包装；它不是直接执行 `add(2, 3)`，也不需要启动端口或另一个 Python 程序。Agent 同样可以连接这种服务。

下面复用同一服务，比较直接调用与 LangChain 工具执行，验证上表中结果的对应关系。我们手工填写一份 ToolCall，验证 Agent 的工具执行环节；这份调用没有经过模型生成。真实模型的调用循环已在 [第二节 2.8](02_MCP原理与工具适配.ipynb) 验证。

In [17]:
from langchain.mcp import MCPAdapter
from langchain_core.messages import ToolMessage

async with Client(calc_server) as direct_client:
    direct_result = await direct_client.call_tool(
        name="add", arguments={"a": 2, "b": 3}, timeout=10
    )

async with MCPAdapter(calc_server) as adapter:
    tools_for_agent = await adapter.list_tools()
    adapted_add = next(item for item in tools_for_agent if item.name == "add")
    message_for_agent = await adapted_add.ainvoke({
        "name": "add",
        "args": {"a": 2, "b": 3},
        "id": "qa_add_1",
        "type": "tool_call",
    })

print("Client 返回:", type(direct_result).__name__, direct_result.data)
print("适配后的工具:", type(adapted_add).__name__)
print("完整 ToolCall 返回 ToolMessage:", isinstance(message_for_agent, ToolMessage))
print("ToolMessage 的文本:", message_for_agent.text)
print("ToolMessage 的调用 ID:", message_for_agent.tool_call_id)
print("ToolMessage 的结构化结果:", message_for_agent.artifact["structured_content"])
print("两条路径的计算结果相同:",
      direct_result.data == message_for_agent.artifact["structured_content"]["result"])

Client 返回: CallToolResult 5
适配后的工具: StructuredTool
完整 ToolCall 返回 ToolMessage: True
ToolMessage 的文本: 5
ToolMessage 的调用 ID: qa_add_1
ToolMessage 的结构化结果: {'result': 5}
两条路径的计算结果相同: True


/var/folders/14/yh05p4nj2z70mjs568rzs9gh0000gn/T/ipykernel_45401/3642840714.py:1: LangChainBetaWarning: `langchain.mcp` is in beta. It is actively being worked on, so the API may change.
  from langchain.mcp import MCPAdapter


`tools_for_agent` 就是可以传给 `create_agent(model=..., tools=tools_for_agent)` 的工具列表。这里的手工 ToolCall 只替代了模型提出请求的步骤；工具适配与 MCP 调用都实际执行了。

### 3.7.2 关键接口的签名、参数和返回值

以下签名以当前环境为准，方法省略 `self`；异步方法的 `->` 表示 **`await` 后得到的值**。调用异步方法而不 `await`，得到的是等待执行的协程。签名中的 `*` 表示后续参数必须使用 `参数名=值`。

先区分同名的“列出工具”：

| 接口及主要签名 | 参数含义 | 返回值 |
|---|---|---|
| `Client(transport, timeout=None, ...)` | `transport` 是连接目标或连接配置，例如服务对象或 stdio Transport；`timeout` 是默认请求超时，数值以秒计 | 构造表达式返回 Client 对象，此时尚未进入连接会话；其余构造选项本节未使用 |
| `await calc_server.list_tools(*, run_middleware=True)` | 服务端查看工具；布尔参数决定是否执行相关中间件 | FastMCP 服务端工具序列，本例元素是 FunctionTool，保存函数实现 |
| `await calc_server.get_tool(name, version=None)` | 按注册名取得一个服务端工具；可指定版本，本例未使用版本功能 | FastMCP 服务端 Tool；本例为 FunctionTool，未找到时为 None |
| `await client.list_tools(max_pages=250, *, cache_mode="use")` | 客户端发现工具；`max_pages` 限制自动分页次数，`cache_mode` 控制已配置的缓存，本例用默认值 | MCP 协议描述的列表 `list[Tool]`，包含名称与 Schema，不包含 Python 业务函数 |
| `await adapter.list_tools(*, cache_mode="use")` | MCPAdapter 发现并适配工具；缓存参数同上 | LangChain 的 `list[BaseTool]`，本例元素是 StructuredTool，可交给 Agent |

`client.list_tools()` 返回列表。前面的 `specs_by_name = {spec.name: spec for spec in tool_specs}` 才是将名称映射到描述对象的代码。

**Client.call_tool 的完整签名**（Any 表示任意 Python 值，timedelta 表示时长，ProgressHandler 是进度回调类型）：

```python
async def call_tool(
    name: str,
    arguments: dict[str, Any] | None = None,
    *,
    version: str | None = None,
    timeout: timedelta | float | int | None = None,
    progress_handler: ProgressHandler | None = None,
    raise_on_error: bool = True,
    meta: dict[str, Any] | None = None,
) -> CallToolResult: ...
```

| 参数 | 含义及本例用法 |
|---|---|
| `name` | 注册的工具名，例如 `"add"`；它不是服务名 `"Calculator"` |
| `arguments` | 业务参数字典，例如 `{"a": 2, "b": 3}`；None 表示没有提供参数，本例必填参数不能省略 |
| `version` | 选择工具版本；本例只有一个版本，使用 None |
| `timeout` | 本次请求的超时，整数或浮点数以秒计，也接受 timedelta；None 使用 Client 的默认配置 |
| `progress_handler` | 接收服务端进度通知的异步回调；本例计算很快且没有上报进度，使用 None |
| `raise_on_error` | 默认 True，工具结果表示失败时抛出 ToolError；False 则保留失败结果供检查。连接失败或超时仍可能抛异常 |
| `meta` | 本次请求的额外元数据；它与 arguments 分开，不会自动成为业务函数参数，本例使用 None |

返回的 `CallToolResult` 是一个**结果对象**，不是函数本身，也不是 LangChain 工具。用加法返回值对照其字段：

| 字段 | 类型 | 本例内容 |
|---|---|---|
| `data` | Any | FastMCP 解析后的 Python 值，本例整数 5；没有可解析的结构化值时可能为 None |
| `content` | 内容块列表 | 本例一个 TextContent，`text` 为字符串 `"5"`；其他工具也可能返回图片等内容 |
| `structured_content` | dict 或 None | 本例 `{"result": 5}` |
| `is_error` | bool | 成功为 False；工具执行失败且未自动抛异常时可检查 True |
| `meta` | dict 或 None | 服务端返回的结果元数据，本例为 None；不保证与请求 meta 相同 |

In [18]:
for field in ["data", "content", "structured_content", "is_error", "meta"]:
    value = getattr(direct_result, field)
    print(f"{field}: 类型={type(value).__name__}, 值={value}")
print("文本内容的类型:", type(direct_result.content[0].text).__name__)

data: 类型=int, 值=5
content: 类型=list, 值=[TextContent(type='text', text='5', annotations=None, meta=None)]
structured_content: 类型=dict, 值={'result': 5}
is_error: 类型=bool, 值=False
meta: 类型=dict, 值={'fastmcp': {'wrap_result': True}, 'io.modelcontextprotocol/serverInfo': {'name': 'Calculator', 'version': '4.1.0'}}
文本内容的类型: str


### 3.7.3 runpy是什么，为什么能取得脚本中的mcp

`runpy` 是 Python 自带的模块。本节使用的函数签名为：

```python
runpy.run_path(path_name, init_globals=None, run_name=None) -> dict
```

| 参数 / 返回值 | 含义 |
|---|---|
| `path_name` | 要执行的脚本路径，本例传 `str(SCRIPT_FILE)` |
| `init_globals` | 执行前放入脚本全局空间的变量字典；本例不需要，使用 None |
| `run_name` | 设置脚本执行时的 `__name__`；默认 None 时为 `"<run_path>"` |
| 返回的 dict | 脚本执行后的全局名称与对象，例如 `{"mcp": FastMCP对象, "add": 函数, ...}`，也包含 `__name__` 等名称 |

它会**在当前进程执行脚本的顶层代码**：导入组件、创建 mcp、定义并注册工具都会发生。默认 `__name__` 不等于 `"__main__"`，所以本脚本的 `mcp.run(...)` 分支没有执行；`script_objects["mcp"]` 取得已经创建的服务对象。

`SCRIPT_FILE.read_text()` 只返回源码字符串，`runpy.run_path(...)` 则执行代码并返回对象字典。这是 Python 文件复用的方法，不是 MCP 协议要求；本节用它复用脚本里的定义，其他项目也可以通过正常 import 组织代码。

### 3.7.4 PythonStdioTransport是什么，与runpy有什么区别

**Transport 是“连接如何建立、数据如何传输”的配置和实现**。`PythonStdioTransport` 专门让客户端启动一个 Python 服务脚本，通过子进程的标准输入和标准输出传递 MCP 消息。

![runpy 与 PythonStdioTransport 的进程和返回对象](assets/12-runpy-and-stdio-transport.svg)

当前构造签名如下。构造表达式返回 **PythonStdioTransport 对象**；此时只创建配置，进入 `async with Client(transport)` 后才启动子进程并建立连接。

```python
PythonStdioTransport(
    script_path,
    args=None,
    env=None,
    cwd=None,
    python_cmd=sys.executable,
    keep_alive=None,
    log_file=None,
)
```

| 参数 | 类型与作用 |
|---|---|
| `script_path` | str 或 Path；需要启动的 `.py` 脚本路径，必须是已存在的文件 |
| `args` | 字符串列表或 None；脚本路径之后附加的命令行参数，本例没有额外参数 |
| `env` | 字符串字典或 None；子进程额外的环境变量，当前实现仅默认继承必要变量，其他配置需显式传入 |
| `cwd` | str 或 None；子进程工作目录，本例为项目根目录；它不是脚本文件名 |
| `python_cmd` | str；用哪个 Python 启动服务，本节显式传 sys.executable 以复用当前环境 |
| `keep_alive` | bool 或 None；是否跨连接保留子进程，当前 None 使用 True；本节显式 False，退出连接时关闭子进程 |
| `log_file` | Path、文本文件对象或 None；接收服务端 stderr 日志，None 时输出到当前 stderr；stdout 仍用于协议通信 |

服务脚本在新 Python 进程中作为主程序执行，`__name__ == "__main__"` 成立，因此启动入口会运行。`mcp.run` 等待请求发生在服务子进程中，notebook 的 Client 通过 stdio 发请求并接收结果。

**用进程 ID 做一个小实验**：临时写一个只报告自身位置的服务，让同一个文件分别经 runpy 和 stdio 执行。PID 是操作系统为进程分配的编号；比较 PID，就能看到“同一进程”与“另一个进程”的区别。退出实验后，客户端关闭子进程，临时文件也自动清理。

In [19]:
import os
from tempfile import TemporaryDirectory
from textwrap import dedent

probe_source = dedent("""
    import os
    from fastmcp import FastMCP

    mcp = FastMCP("ProcessProbe")

    @mcp.tool
    def where_am_i() -> dict:
        "返回执行此工具的进程编号与脚本名称。"
        return {"pid": os.getpid(), "module_name": __name__}

    if __name__ == "__main__":
        mcp.run(transport="stdio", show_banner=False)
""")
# 临时创建文件目录
with TemporaryDirectory(prefix="mcp-process-probe-") as temporary_directory:
    probe_path = Path(temporary_directory) / "process_probe.py"
    # 将代码写入文件
    probe_path.write_text(probe_source, encoding="utf-8")
    # 在当前进程运行脚本文件
    probe_objects = runpy.run_path(str(probe_path))

    async with Client(probe_objects["mcp"]) as inprocess_client:
        # 连接服务端,然后客户端运行
        inprocess_location = await inprocess_client.call_tool("where_am_i")
    # 返回连接配置
    probe_transport = PythonStdioTransport(
        script_path=probe_path,
        python_cmd=sys.executable,
        cwd=str(PROJECT_ROOT),
        keep_alive=False,
    )
    print("Transport 构造返回:", type(probe_transport).__name__)
    async with Client(probe_transport, timeout=15) as subprocess_client:
        subprocess_location = await subprocess_client.call_tool("where_am_i")

print("notebook PID:", os.getpid())
print("runpy 返回:", type(probe_objects).__name__)
print("runpy 取得的 mcp:", type(probe_objects["mcp"]).__name__)
print("进程内工具结果:", inprocess_location.data)
print("子进程工具结果:", subprocess_location.data)
print("runpy 路径的 PID 等于 notebook:", inprocess_location.data["pid"] == os.getpid())
print("stdio 路径的 PID 不同于 notebook:", subprocess_location.data["pid"] != os.getpid())

Transport 构造返回: PythonStdioTransport


notebook PID: 45401
runpy 返回: dict
runpy 取得的 mcp: FastMCP
进程内工具结果: {'pid': 45401, 'module_name': '<run_path>'}
子进程工具结果: {'pid': 45534, 'module_name': '__main__'}
runpy 路径的 PID 等于 notebook: True
stdio 路径的 PID 不同于 notebook: True


结果应显示：进程内工具的 PID 与 notebook 相同，名称是 `<run_path>`；stdio 工具的 PID 不同，名称是 `__main__`。

把三类返回值对齐后，原来的代码就能逐行读懂：

| 表达式 | 得到什么 | 下一步怎么使用 |
|---|---|---|
| `runpy.run_path(...)` | 全局对象字典 | 取出 `["mcp"]`，交给 Client 做进程内连接 |
| `PythonStdioTransport(...)` | 子进程连接配置对象 | 交给 Client，连接时启动并管理服务进程 |
| `await client.call_tool(...)` | CallToolResult | 读取 data、content、structured_content 和状态 |
| `await adapter.list_tools()` | LangChain 工具列表 | 交给 Agent，由工具包装器使用 Client 请求服务 |

参考：[LangChain MCP 的 Agent 接入](https://docs.langchain.com/oss/python/langchain/mcp)、[FastMCP 客户端工具调用](https://gofastmcp.com/clients/tools)、[客户端 Transport](https://gofastmcp.com/clients/transports)、[Python runpy 文档](https://docs.python.org/3.11/library/runpy.html)。

## 3.8 小结与小练习

| 步骤 | 本节观察到的现象 |
|---|---|
| 创建服务 | `FastMCP` 组织一组工具，服务名与工具名分别使用 |
| 注册工具 | `@mcp.tool` 生成服务端工具对象，保存业务函数与接口信息 |
| 发现工具 | `Client.list_tools()` 返回协议描述，不执行计算 |
| 生成 Schema | 类型、默认值与 `Field` 约束决定客户端如何填写参数 |
| 调用工具 | 校验参数 → 执行函数 → 包装成功结果或工具错误 |
| 独立运行 | 工具定义保存在脚本，启动入口由客户端通过子进程执行 |

小练习：

1. 把 3.4.2 的精度改成 0 和 6，再改成 -1。预测结果与函数进入次数，再验证。
2. 新建一个服务，用 `@server.tool(name="sum_two_numbers")` 注册名为 `add` 的函数。比较函数名、客户端发现的工具名与 Schema；调用时使用新工具名。
3. 在独立脚本中增加 `multiply(a: int, b: int) -> int`，用进程内客户端先验证，再让子进程客户端发现并调用。增加工具后，更新 3.6.3 的预期工具集合。

下一节复用本节的 [计算服务脚本](servers/calculator_server.py)，比较 stdio 与 Streamable HTTP 的连接和生命周期。

参考：[FastMCP 工具定义](https://gofastmcp.com/servers/tools)、[客户端连接方式](https://gofastmcp.com/clients/transports)、[服务启动示例](https://gofastmcp.com/getting-started/quickstart)。代码与结果以本节打印的实际依赖版本为准。